
## Folder 07 — File 07: Gold Executive KPIs:



- This is the final notebook in the Gold business analytics folder. 
- It combines the main business metrics into an executive-level summary.


It covers:

- Total orders and order value.
- Unique purchasing customers.
- Average order value.
- Successful payments and net payment amount.
- Payment success rate.
- Total shipments and delivered shipments.
- Delivery rate.
- Total inventory and stock valuation.
- Low-stock and out-of-stock product counts.
- Executive KPI status by currency.

**Important design decision:**

- The executive table will have one row per currency. 
- Sales and payment amounts must never be added across different currencies. 
- The supplied project data uses INR, but the notebook keeps the currency dimension explicit.

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # 07.7 - Gold Executive KPIs
# MAGIC
# MAGIC Consolidate sales, customer, payment, shipping and inventory
# MAGIC metrics into an executive-level KPI table.
# MAGIC
# MAGIC Grain: one row per currency.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Import libraries

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Project configuration

# COMMAND ----------

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

ORDERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
CUSTOMERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
PAYMENTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
SHIPMENTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
INVENTORY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_inventory"
PRODUCTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_products"

GOLD_EXECUTIVE_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_executive_kpis"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

LOW_STOCK_THRESHOLD = 10

# This assumes product costs and prices in the supplied
# inventory/product data are denominated in INR.
INVENTORY_CURRENCY = "INR"

print("Run ID:", RUN_ID)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Create Gold schema and validate source tables

# COMMAND ----------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    ORDERS_TABLE,
    CUSTOMERS_TABLE,
    PAYMENTS_TABLE,
    SHIPMENTS_TABLE,
    INVENTORY_TABLE,
    PRODUCTS_TABLE
]

for table_name in SOURCE_TABLES:

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Missing source table: {table_name}"
        )

print("All required Silver tables are available.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Load Silver datasets

# COMMAND ----------

orders_raw_df = spark.table(ORDERS_TABLE)

customers_raw_df = spark.table(CUSTOMERS_TABLE)

payments_raw_df = spark.table(PAYMENTS_TABLE)

shipments_raw_df = spark.table(SHIPMENTS_TABLE)

inventory_raw_df = spark.table(INVENTORY_TABLE)

products_raw_df = spark.table(PRODUCTS_TABLE)

print("Source datasets loaded successfully.")

print("Orders:", orders_raw_df.count())
print("Customers:", customers_raw_df.count())
print("Payments:", payments_raw_df.count())
print("Shipments:", shipments_raw_df.count())
print("Inventory:", inventory_raw_df.count())
print("Products:", products_raw_df.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Validate required columns

# COMMAND ----------

required_columns = {

    ORDERS_TABLE: [
        "order_id",
        "customer_id",
        "order_timestamp",
        "order_status",
        "currency",
        "order_total"
    ],

    CUSTOMERS_TABLE: [
        "customer_id"
    ],

    PAYMENTS_TABLE: [
        "payment_id",
        "order_id",
        "payment_status",
        "amount",
        "payment_timestamp"
    ],

    SHIPMENTS_TABLE: [
        "shipment_id",
        "order_id",
        "shipment_status"
    ],

    INVENTORY_TABLE: [
        "inventory_snapshot_id",
        "product_id",
        "warehouse_id",
        "snapshot_date",
        "quantity_on_hand"
    ],

    PRODUCTS_TABLE: [
        "product_id",
        "unit_cost",
        "unit_price"
    ]
}

source_dataframes = {

    ORDERS_TABLE: orders_raw_df,
    CUSTOMERS_TABLE: customers_raw_df,
    PAYMENTS_TABLE: payments_raw_df,
    SHIPMENTS_TABLE: shipments_raw_df,
    INVENTORY_TABLE: inventory_raw_df,
    PRODUCTS_TABLE: products_raw_df
}

for table_name, columns in required_columns.items():

    actual_columns = source_dataframes[
        table_name
    ].columns

    for column_name in columns:

        if column_name not in actual_columns:

            raise Exception(
                f"Missing column {column_name} "
                f"in {table_name}"
            )

print("All required columns are available.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Define deterministic deduplication helper

# MAGIC
# MAGIC Prefer the latest ingestion timestamp when available.
# MAGIC Otherwise, use the business timestamp and a stable hash
# MAGIC as a deterministic tie-breaker.

# COMMAND ----------

def deduplicate_latest(
    dataframe,
    key_columns,
    business_timestamp_column=None
):

    ordering_columns = []

    if "_ingestion_timestamp" in dataframe.columns:

        ordering_columns.append(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )

    elif (
        business_timestamp_column is not None
        and business_timestamp_column in dataframe.columns
    ):

        ordering_columns.append(
            F.col(business_timestamp_column).desc_nulls_last()
        )

    else:

        print(
            "WARNING: No reliable timestamp available for:",
            key_columns
        )

    stable_hash = F.sha2(
        F.to_json(
            F.struct(
                *[
                    F.col(column_name)
                    for column_name in dataframe.columns
                ]
            )
        ),
        256
    )

    ordering_columns.append(
        stable_hash.desc()
    )

    deduplication_window = (
        Window
        .partitionBy(*key_columns)
        .orderBy(*ordering_columns)
    )

    result_df = (
        dataframe
        .withColumn(
            "_row_number",
            F.row_number().over(deduplication_window)
        )
        .filter(
            F.col("_row_number") == 1
        )
        .drop("_row_number")
    )

    return result_df


# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Clean and deduplicate orders

# COMMAND ----------

orders_clean_df = (
    orders_raw_df

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )

    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp")
    )

    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )

    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )

    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )

    .withColumn(
        "order_total",
        F.col("order_total").cast("decimal(18,2)")
    )

    .filter(
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("currency").isNotNull()
        & (F.col("currency") != "")
        & F.col("order_timestamp").isNotNull()
        & F.col("order_total").isNotNull()
        & (F.col("order_total") >= 0)
    )
)

orders_df = deduplicate_latest(
    orders_clean_df,
    ["order_id"],
    "order_timestamp"
)

print("Deduplicated orders:", orders_df.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Calculate sales and customer KPIs

# MAGIC
# MAGIC Cancelled orders remain included in total order counts.
# MAGIC They are excluded from sales value and purchasing customers.

# COMMAND ----------

sales_kpis_df = (

    orders_df

    .groupBy("currency")

    .agg(

        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.countDistinct(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_id")
            )
        ).alias(
            "non_cancelled_orders"
        ),

        F.countDistinct(
            F.when(
                F.col("order_status") == "CANCELLED",
                F.col("order_id")
            )
        ).alias(
            "cancelled_orders"
        ),

        F.countDistinct(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("customer_id")
            )
        ).alias(
            "unique_purchasing_customers"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_total")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "gross_order_value"
        ),

        F.avg(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_total")
            )
        ).cast("decimal(18,2)").alias(
            "average_order_value"
        )
    )

    .withColumn(
        "order_cancellation_rate",

        F.when(
            F.col("total_orders") > 0,

            F.round(
                F.col("cancelled_orders")
                / F.col("total_orders") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Clean and deduplicate payment records

# COMMAND ----------

payments_clean_df = (

    payments_raw_df

    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )

    .withColumn(
        "amount",
        F.col("amount").cast("decimal(18,2)")
    )

    .withColumn(
        "payment_timestamp",
        F.to_timestamp("payment_timestamp")
    )

    .filter(
        F.col("payment_id").isNotNull()
        & (F.col("payment_id") != "")
        & F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("payment_status").isin(
            "SUCCESS",
            "FAILED",
            "PENDING",
            "REFUNDED"
        )
        & F.col("amount").isNotNull()
        & (F.col("amount") >= 0)
    )
)

payments_df = deduplicate_latest(
    payments_clean_df,
    ["payment_id"],
    "payment_timestamp"
)

print("Deduplicated payments:", payments_df.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Aggregate payments by order

# MAGIC
# MAGIC Aggregate payment attempts before joining to orders.
# MAGIC This prevents one-to-many payment joins from multiplying
# MAGIC order values.

# COMMAND ----------

payment_by_order_df = (

    payments_df

    .groupBy("order_id")

    .agg(

        F.countDistinct("payment_id").alias(
            "total_payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                1
            ).otherwise(0)
        ).alias(
            "successful_payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                1
            ).otherwise(0)
        ).alias(
            "pending_payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                1
            ).otherwise(0)
        ).alias(
            "refunded_payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "successful_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "refunded_payment_amount"
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Calculate payment KPIs by currency

# MAGIC
# MAGIC Payment aggregates are assigned to the currency of their
# MAGIC associated order.
# MAGIC Payments without a matching valid order are excluded.

# COMMAND ----------

payment_currency_df = (

    orders_df.alias("orders")

    .join(
        payment_by_order_df.alias("payments"),

        F.col("orders.order_id")
        == F.col("payments.order_id"),

        "inner"
    )

    .select(

        F.col("orders.currency"),

        F.col("payments.total_payment_attempts"),

        F.col("payments.successful_payment_attempts"),

        F.col("payments.failed_payment_attempts"),

        F.col("payments.pending_payment_attempts"),

        F.col("payments.refunded_payment_attempts"),

        F.col("payments.successful_payment_amount"),

        F.col("payments.refunded_payment_amount")
    )
)

payment_kpis_df = (

    payment_currency_df

    .groupBy("currency")

    .agg(

        F.sum("total_payment_attempts").alias(
            "total_payment_attempts"
        ),

        F.sum("successful_payment_attempts").alias(
            "successful_payment_attempts"
        ),

        F.sum("failed_payment_attempts").alias(
            "failed_payment_attempts"
        ),

        F.sum("pending_payment_attempts").alias(
            "pending_payment_attempts"
        ),

        F.sum("refunded_payment_attempts").alias(
            "refunded_payment_attempts"
        ),

        F.sum("successful_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "successful_payment_amount"
        ),

        F.sum("refunded_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "refunded_payment_amount"
        )
    )

    .withColumn(
        "net_payment_amount",

        (
            F.col("successful_payment_amount")
            - F.col("refunded_payment_amount")
        ).cast("decimal(18,2)")
    )

    .withColumn(
        "payment_success_rate",

        F.when(
            F.col("total_payment_attempts") > 0,

            F.round(
                F.col("successful_payment_attempts")
                / F.col("total_payment_attempts") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "payment_failure_rate",

        F.when(
            F.col("total_payment_attempts") > 0,

            F.round(
                F.col("failed_payment_attempts")
                / F.col("total_payment_attempts") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Calculate orders without payment attempts

# MAGIC
# MAGIC Use a LEFT ANTI JOIN to identify orders that have no payment record.
# MAGIC Do not subtract payment attempt counts from order counts.

# COMMAND ----------

orders_without_payment_df = (

    orders_df

    .filter(
        F.col("order_status") != "CANCELLED"
    )

    .join(
        payment_by_order_df.select("order_id"),

        on="order_id",

        how="left_anti"
    )

)

orders_without_payment_kpis_df = (

    orders_without_payment_df

    .groupBy("currency")

    .agg(

        F.countDistinct("order_id").alias(
            "orders_without_payment_attempts"
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. Clean and deduplicate shipments

# MAGIC
# MAGIC Include RETURN_TO_ORIGIN as a separate shipment status.
# MAGIC Do not silently discard it or treat it as RETURNED.

# COMMAND ----------

shipments_clean_df = (

    shipments_raw_df

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "shipment_status",
        F.upper(F.trim(F.col("shipment_status")))
    )

    .filter(
        F.col("shipment_id").isNotNull()
        & (F.col("shipment_id") != "")
        & F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("shipment_status").isin(
            "PENDING",
            "PROCESSING",
            "SHIPPED",
            "IN_TRANSIT",
            "OUT_FOR_DELIVERY",
            "DELIVERED",
            "CANCELLED",
            "RETURNED",
            "RETURN_TO_ORIGIN"
        )
    )
)

shipments_df = deduplicate_latest(
    shipments_clean_df,
    ["shipment_id"],
    None
)

print("Deduplicated shipments:", shipments_df.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 14. Aggregate shipments by order

# MAGIC
# MAGIC Multiple shipments can belong to one order.
# MAGIC Aggregate shipment records before joining with orders.

# COMMAND ----------

shipment_by_order_df = (

    shipments_df

    .groupBy("order_id")

    .agg(

        F.countDistinct("shipment_id").alias(
            "total_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "IN_TRANSIT",
                1
            ).otherwise(0)
        ).alias(
            "in_transit_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "RETURNED",
                1
            ).otherwise(0)
        ).alias(
            "returned_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "RETURN_TO_ORIGIN",
                1
            ).otherwise(0)
        ).alias(
            "return_to_origin_shipments"
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 15. Calculate shipping KPIs by currency

# MAGIC
# MAGIC Shipment counts are grouped using the currency of the associated order.

# COMMAND ----------

shipment_currency_df = (

    orders_df.alias("orders")

    .join(
        shipment_by_order_df.alias("shipments"),

        F.col("orders.order_id")
        == F.col("shipments.order_id"),

        "inner"
    )

    .select(

        F.col("orders.currency"),

        F.col("shipments.total_shipments"),

        F.col("shipments.delivered_shipments"),

        F.col("shipments.in_transit_shipments"),

        F.col("shipments.cancelled_shipments"),

        F.col("shipments.returned_shipments"),

        F.col("shipments.return_to_origin_shipments")
    )
)

shipping_kpis_df = (

    shipment_currency_df

    .groupBy("currency")

    .agg(

        F.sum("total_shipments").alias(
            "total_shipments"
        ),

        F.sum("delivered_shipments").alias(
            "delivered_shipments"
        ),

        F.sum("in_transit_shipments").alias(
            "in_transit_shipments"
        ),

        F.sum("cancelled_shipments").alias(
            "cancelled_shipments"
        ),

        F.sum("returned_shipments").alias(
            "returned_shipments"
        ),

        F.sum("return_to_origin_shipments").alias(
            "return_to_origin_shipments"
        )
    )

    .withColumn(
        "delivery_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("delivered_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 16. Clean inventory records

# MAGIC
# MAGIC Use the latest inventory snapshot per product and warehouse.
# MAGIC Do not sum historical snapshots.

# COMMAND ----------

inventory_clean_df = (

    inventory_raw_df

    .withColumn(
        "inventory_snapshot_id",
        F.trim(F.col("inventory_snapshot_id"))
    )

    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )

    .withColumn(
        "warehouse_id",
        F.upper(F.trim(F.col("warehouse_id")))
    )

    .withColumn(
        "quantity_on_hand",
        F.col("quantity_on_hand").cast("int")
    )

    .withColumn(
        "snapshot_date",
        F.to_date("snapshot_date")
    )

    .filter(
        F.col("inventory_snapshot_id").isNotNull()
        & (F.col("inventory_snapshot_id") != "")
        & F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("warehouse_id").isNotNull()
        & (F.col("warehouse_id") != "")
        & F.col("quantity_on_hand").isNotNull()
        & (F.col("quantity_on_hand") >= 0)
        & F.col("snapshot_date").isNotNull()
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 17. Deduplicate inventory snapshots and select latest inventory

# COMMAND ----------

inventory_business_window = (

    Window

    .partitionBy(
        "product_id",
        "warehouse_id",
        "snapshot_date"
    )

    .orderBy(
        F.col("inventory_snapshot_id").desc()
    )
)

inventory_deduplicated_df = (

    inventory_clean_df

    .withColumn(
        "_row_number",
        F.row_number().over(inventory_business_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)

latest_inventory_window = (

    Window

    .partitionBy(
        "product_id",
        "warehouse_id"
    )

    .orderBy(
        F.col("snapshot_date").desc(),
        F.col("inventory_snapshot_id").desc()
    )
)

latest_inventory_df = (

    inventory_deduplicated_df

    .withColumn(
        "_row_number",
        F.row_number().over(latest_inventory_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)

print(
    "Latest product-warehouse inventory records:",
    latest_inventory_df.count()
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 18. Clean and deduplicate products

# COMMAND ----------

products_clean_df = (

    products_raw_df

    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )

    .withColumn(
        "unit_cost",
        F.col("unit_cost").cast("decimal(18,2)")
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(18,2)")
    )

    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
    )
)

products_deduplicated_df = deduplicate_latest(
    products_clean_df,
    ["product_id"],
    None
)

print(
    "Deduplicated products:",
    products_deduplicated_df.count()
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 19. Calculate inventory KPIs

# MAGIC
# MAGIC Inventory is not associated with an order currency.
# MAGIC The supplied project data uses INR, so inventory values are reported
# MAGIC only on the INR executive KPI row.

# COMMAND ----------

inventory_product_df = (

    latest_inventory_df

    .groupBy("product_id")

    .agg(

        F.sum("quantity_on_hand").alias(
            "total_quantity_on_hand"
        ),

        F.max("snapshot_date").alias(
            "latest_snapshot_date"
        )
    )
)

inventory_with_products_df = (

    products_deduplicated_df.alias("products")

    .join(
        inventory_product_df.alias("inventory"),

        F.col("products.product_id")
        == F.col("inventory.product_id"),

        "left"
    )

    .select(

        F.col("products.product_id"),

        F.col("products.unit_cost"),

        F.col("products.unit_price"),

        F.coalesce(
            F.col("inventory.total_quantity_on_hand"),
            F.lit(0)
        ).alias(
            "total_quantity_on_hand"
        ),

        F.col("inventory.latest_snapshot_date")
    )
)

inventory_kpis_df = (

    inventory_with_products_df

    .agg(

        F.countDistinct("product_id").alias(
            "total_products"
        ),

        F.sum("total_quantity_on_hand").alias(
            "total_inventory_units"
        ),

        F.sum(
            F.when(
                F.col("total_quantity_on_hand") == 0,
                1
            ).otherwise(0)
        ).alias(
            "out_of_stock_products"
        ),

        F.sum(
            F.when(
                (
                    F.col("total_quantity_on_hand") > 0
                )
                & (
                    F.col("total_quantity_on_hand")
                    <= LOW_STOCK_THRESHOLD
                ),
                1
            ).otherwise(0)
        ).alias(
            "low_stock_products"
        ),

        F.sum(
            F.when(
                F.col("total_quantity_on_hand") > 0,
                1
            ).otherwise(0)
        ).alias(
            "available_products"
        ),

        F.sum(
            F.when(
                F.col("unit_cost").isNotNull(),

                F.col("total_quantity_on_hand")
                * F.col("unit_cost")
            )
        ).cast("decimal(18,2)").alias(
            "inventory_cost_value"
        ),

        F.sum(
            F.when(
                F.col("unit_price").isNotNull(),

                F.col("total_quantity_on_hand")
                * F.col("unit_price")
            )
        ).cast("decimal(18,2)").alias(
            "inventory_retail_value"
        ),

        F.max("latest_snapshot_date").alias(
            "latest_inventory_snapshot_date"
        )
    )

    .withColumn(
        "inventory_availability_rate",

        F.when(
            F.col("total_products") > 0,

            F.round(
                F.col("available_products")
                / F.col("total_products") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 20. Combine all business KPIs

# MAGIC
# MAGIC Start with sales by currency.
# MAGIC Join payment, missing-payment and shipping metrics by currency.
# MAGIC Inventory is added as a single-row aggregate.

# COMMAND ----------

executive_kpis_df = (

    sales_kpis_df.alias("sales")

    .join(
        payment_kpis_df.alias("payments"),

        on="currency",

        how="left"
    )

    .join(
        orders_without_payment_kpis_df.alias("missing_payments"),

        on="currency",

        how="left"
    )

    .join(
        shipping_kpis_df.alias("shipping"),

        on="currency",

        how="left"
    )

    .crossJoin(
        inventory_kpis_df
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 21. Handle missing payment and shipping KPI values

# MAGIC
# MAGIC Missing payment or shipping activity is represented as zero.

# COMMAND ----------

zero_fill_columns = [

    "total_payment_attempts",
    "successful_payment_attempts",
    "failed_payment_attempts",
    "pending_payment_attempts",
    "refunded_payment_attempts",
    "successful_payment_amount",
    "refunded_payment_amount",
    "net_payment_amount",
    "payment_success_rate",
    "payment_failure_rate",
    "orders_without_payment_attempts",

    "total_shipments",
    "delivered_shipments",
    "in_transit_shipments",
    "cancelled_shipments",
    "returned_shipments",
    "return_to_origin_shipments",
    "delivery_rate"
]

for column_name in zero_fill_columns:

    if column_name in executive_kpis_df.columns:

        executive_kpis_df = (
            executive_kpis_df.withColumn(
                column_name,

                F.coalesce(
                    F.col(column_name),
                    F.lit(0)
                )
            )
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 22. Add currency-specific inventory values

# MAGIC
# MAGIC Inventory valuation is reported only for INR.
# MAGIC Other currencies receive NULL rather than misleading converted values.

# COMMAND ----------

inventory_columns = [

    "total_products",
    "total_inventory_units",
    "out_of_stock_products",
    "low_stock_products",
    "available_products",
    "inventory_cost_value",
    "inventory_retail_value",
    "inventory_availability_rate",
    "latest_inventory_snapshot_date"
]

for column_name in inventory_columns:

    executive_kpis_df = (

        executive_kpis_df.withColumn(

            column_name,

            F.when(
                F.col("currency") == INVENTORY_CURRENCY,
                F.col(column_name)
            ).otherwise(
                F.lit(None).cast(
                    executive_kpis_df.schema[column_name].dataType
                )
            )
        )
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 23. Add executive-level indicators

# COMMAND ----------

executive_kpis_df = (

    executive_kpis_df

    .withColumn(
        "payment_collection_vs_order_value_rate",

        F.when(
            F.col("gross_order_value") > 0,

            F.round(
                F.col("net_payment_amount")
                / F.col("gross_order_value") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "executive_payment_status",

        F.when(
            F.col("total_payment_attempts") == 0,
            "NO_PAYMENT_ACTIVITY"
        )

        .when(
            F.col("payment_success_rate") >= 90,
            "GOOD"
        )

        .when(
            F.col("payment_success_rate") >= 70,
            "MONITOR"
        )

        .otherwise("NEEDS_ATTENTION")
    )

    .withColumn(
        "executive_delivery_status",

        F.when(
            F.col("total_shipments") == 0,
            "NO_SHIPPING_ACTIVITY"
        )

        .when(
            F.col("delivery_rate") >= 90,
            "GOOD"
        )

        .when(
            F.col("delivery_rate") >= 70,
            "MONITOR"
        )

        .otherwise("NEEDS_ATTENTION")
    )

    .withColumn(
        "executive_inventory_status",

        F.when(
            F.col("currency") != INVENTORY_CURRENCY,
            "NOT_AVAILABLE_FOR_CURRENCY"
        )

        .when(
            F.col("out_of_stock_products") > 0,
            "OUT_OF_STOCK_ITEMS_PRESENT"
        )

        .when(
            F.col("low_stock_products") > 0,
            "LOW_STOCK_ITEMS_PRESENT"
        )

        .otherwise("HEALTHY")
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 24. Add audit columns

# COMMAND ----------

executive_kpis_df = (

    executive_kpis_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "generated_at",
        F.lit(RUN_TIMESTAMP)
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 25. Select final executive KPI columns

# COMMAND ----------

final_executive_kpis_df = executive_kpis_df.select(

    "currency",

    # Sales KPIs
    "total_orders",
    "non_cancelled_orders",
    "cancelled_orders",
    "unique_purchasing_customers",
    "gross_order_value",
    "average_order_value",
    "order_cancellation_rate",

    # Payment KPIs
    "total_payment_attempts",
    "successful_payment_attempts",
    "failed_payment_attempts",
    "pending_payment_attempts",
    "refunded_payment_attempts",
    "successful_payment_amount",
    "refunded_payment_amount",
    "net_payment_amount",
    "payment_success_rate",
    "payment_failure_rate",
    "payment_collection_vs_order_value_rate",
    "orders_without_payment_attempts",

    # Shipping KPIs
    "total_shipments",
    "delivered_shipments",
    "in_transit_shipments",
    "cancelled_shipments",
    "returned_shipments",
    "return_to_origin_shipments",
    "delivery_rate",

    # Inventory KPIs
    "total_products",
    "total_inventory_units",
    "available_products",
    "low_stock_products",
    "out_of_stock_products",
    "inventory_cost_value",
    "inventory_retail_value",
    "inventory_availability_rate",
    "latest_inventory_snapshot_date",

    # Executive indicators
    "executive_payment_status",
    "executive_delivery_status",
    "executive_inventory_status",

    # Audit
    "run_id",
    "generated_at"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 26. Display executive KPI summary

# COMMAND ----------

display(
    final_executive_kpis_df.orderBy("currency")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 27. Validate KPI grain before saving

# COMMAND ----------

duplicate_currency_df = (

    final_executive_kpis_df

    .groupBy("currency")

    .count()

    .filter(
        F.col("count") > 1
    )
)

duplicate_currency_count = duplicate_currency_df.count()

if duplicate_currency_count > 0:

    display(duplicate_currency_df)

    raise Exception(
        "Executive KPI output contains duplicate currency rows."
    )

null_currency_count = (

    final_executive_kpis_df

    .filter(
        F.col("currency").isNull()
        | (F.col("currency") == "")
    )

    .count()
)

if null_currency_count > 0:

    raise Exception(
        "Executive KPI output contains null or empty currency."
    )

print("Currency grain validation passed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 28. Save Gold executive KPI table

# COMMAND ----------

(
    final_executive_kpis_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_EXECUTIVE_TABLE
    )
)

print(
    "Executive KPI table created:",
    GOLD_EXECUTIVE_TABLE
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 29. Validate Gold executive KPI table

# COMMAND ----------

gold_executive_check_df = spark.table(
    GOLD_EXECUTIVE_TABLE
)

gold_record_count = gold_executive_check_df.count()

expected_record_count = (
    final_executive_kpis_df.count()
)

print(
    "Expected executive KPI rows:",
    expected_record_count
)

print(
    "Actual executive KPI rows:",
    gold_record_count
)

print(
    "Executive KPI columns:",
    len(gold_executive_check_df.columns)
)

if gold_record_count != expected_record_count:

    raise Exception(
        "Gold executive KPI record count validation failed."
    )

if gold_executive_check_df.columns != final_executive_kpis_df.columns:

    raise Exception(
        "Gold executive KPI schema validation failed."
    )

print("Gold table record count and schema validation passed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 30. Final KPI validation

# COMMAND ----------

display(
    gold_executive_check_df.select(

        "currency",

        "total_orders",

        "gross_order_value",

        "average_order_value",

        "unique_purchasing_customers",

        "net_payment_amount",

        "payment_success_rate",

        "payment_collection_vs_order_value_rate",

        "orders_without_payment_attempts",

        "total_shipments",

        "delivered_shipments",

        "return_to_origin_shipments",

        "delivery_rate",

        "total_inventory_units",

        "low_stock_products",

        "out_of_stock_products",

        "inventory_cost_value",

        "executive_payment_status",

        "executive_delivery_status",

        "executive_inventory_status"
    )

    .orderBy("currency")
)


## 31. Final execution summary

print("=" * 70)
print("GOLD EXECUTIVE KPIs COMPLETED")
print("=" * 70)

print("Executive KPI table:", GOLD_EXECUTIVE_TABLE)

print("Run ID:", RUN_ID)

print("Executive KPI rows:", gold_record_count)

print("Executive KPI columns:", len(gold_executive_check_df.columns))

print("=" * 70)


## 3. Databricks SQL dashboard queries:


### Executive overview

In [0]:
%sql

SELECT
    currency,
    total_orders,
    gross_order_value,
    average_order_value,
    unique_purchasing_customers,
    net_payment_amount,
    payment_success_rate,
    delivery_rate
FROM workspace.ecommerce_gold.gold_executive_kpis;


### Sales and payment comparison

In [0]:
%sql

SELECT
    currency,
    gross_order_value,
    successful_payment_amount,
    refunded_payment_amount,
    net_payment_amount,
    payment_collection_vs_order_value_rate
FROM workspace.ecommerce_gold.gold_executive_kpis;


### Shipping performance

In [0]:
%sql

SELECT
    currency,
    total_shipments,
    delivered_shipments,
    in_transit_shipments,
    delivery_rate,
    executive_delivery_status
FROM workspace.ecommerce_gold.gold_executive_kpis;


### Inventory health

In [0]:
%sql

SELECT
    currency,
    total_products,
    total_inventory_units,
    available_products,
    low_stock_products,
    out_of_stock_products,
    inventory_cost_value,
    inventory_retail_value,
    executive_inventory_status
FROM workspace.ecommerce_gold.gold_executive_kpis;


### Executive status

In [0]:
%sql

SELECT
    currency,
    executive_payment_status,
    executive_delivery_status,
    executive_inventory_status,
    generated_at
FROM workspace.ecommerce_gold.gold_executive_kpis;

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_gold;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_warehouse_inventory;